In [ ]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3

# %matplotlib qt
%matplotlib inline
import matplotlib.pyplot as plt

from pathlib import Path
import spikeinterface.extractors as se
import spikeinterface.full as si
from neuropy.core.session.init_from_raw_data import windows_to_wsl_path_if_needed, find_first_file_rglob


In [ ]:
## Bapun Format:
# basedir = Path('/home/halechr/FastData/Bapun/RatS/Day1Openfield') # Linux
# basedir = Path('/nfs/turbo/umms-kdiba/Data/Bapun/RatS/Day1Openfield') # Greatlakes
# basedir = windows_to_wsl_path_if_needed(Path('W:/Data/Bapun/RatS/Day1Openfield')) # Windows
# basedir = Path("/mnt/w/Data/Bapun/RatS/Day1Openfield") # Apogee WSL
# # n_channels: int = 200
# n_channels: int = 195
# dat_file_sampling_rate: int = 30000
# eeg_sampling_rate: int = 1250
# basename: str = 'RatS-Day1Openfield'
# excluded_data_datetimes = ['2020-11-25_10-24-24', '2020-11-25_15-06-02']


# basedir = Path('/home/halechr/FastData/Bapun/RatS/Day4Openfield') # Linux
basedir = Path('/nfs/turbo/umms-kdiba/Data/Bapun/RatS/Day4Openfield') # Greatlakes
# basedir = windows_to_wsl_path_if_needed(Path('W:/Data/Bapun/RatS/Day4Openfield')) # Windows
# basedir = Path("/mnt/w/Data/Bapun/RatS/Day4Openfield") # Apogee WSL
n_channels: int = 195
dat_file_sampling_rate: int = 30000
eeg_sampling_rate: int = 1250
basename: str = 'RatS-Day4Openfield'
excluded_data_datetimes = []

In [ ]:
xml_path: Path = find_first_file_rglob(basedir, '*.xml', recursive=False)
xml_path
print(f'xml_path: {xml_path}')

concatenated_dat_file: Path = windows_to_wsl_path_if_needed(basedir.joinpath("spyk-circ", f"{basename}.dat").resolve())
print(f"concatenated_dat_file: {concatenated_dat_file.as_posix()}")
assert concatenated_dat_file.exists(), f"concatenated_dat_file: {concatenated_dat_file} does not exist."


In [ ]:
spiking_circus_dir: Path = windows_to_wsl_path_if_needed(basedir.joinpath('spyk-circ', basename).resolve())
assert spiking_circus_dir.exists(), f"spiking_circus_dir: {spiking_circus_dir} does not exist!"
# 1. Map the raw binary file (doesn't load into RAM yet)
# se.read_phy
spiking_circus_loaded = se.read_spykingcircus(spiking_circus_dir)

spiking_circus_loaded

In [ ]:
phy_gui_dir: Path = windows_to_wsl_path_if_needed(
    basedir.joinpath('spyk-circ', basename, f'{basename}-merged.GUI').resolve()
)
assert phy_gui_dir.exists(), f"phy_gui_dir: {phy_gui_dir} does not exist!"

In [ ]:
sorting = se.read_phy(phy_gui_dir)
sorting  # PhySortingExtractor: 138 units - 1 segments - 30.0kHz

In [ ]:
# spiking_circus_loaded

si.create_sorting_analyzer()

In [ ]:
n_channels

In [ ]:
recording = se.read_binary(
    file_paths=concatenated_dat_file.as_posix(),
    sampling_frequency=dat_file_sampling_rate,
    num_channels=n_channels,
    dtype="int16",
)
gain_to_uV = 0.19499999284744263  # from structure.oebin / settings.xml
recording.set_channel_gains(gain_to_uV)
recording.set_channel_offsets(0)
recording


In [ ]:
from probeinterface.io import read_prb

prb_path: Path = windows_to_wsl_path_if_needed(basedir.joinpath("spyk-circ", f"{basename}.prb").resolve())
probe = read_prb(prb_path).probes[0]
recording = recording.set_probe(probe, in_place=False)
recording_filtered = si.bandpass_filter(recording)
recording_filtered


In [ ]:
sorting = spiking_circus_loaded

job_kwargs = dict(n_jobs=-1, progress_bar=True, chunk_duration="1s")

sorting_analyzer_folder: Path = windows_to_wsl_path_if_needed(basedir.joinpath("spyk-circ", f"{basename}-sorting_analyzer").resolve())
sorting_analyzer = si.create_sorting_analyzer(sorting, recording_filtered, format="binary_folder", folder=sorting_analyzer_folder.as_posix())
sorting_analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=500)
sorting_analyzer.compute("waveforms", **job_kwargs)
sorting_analyzer.compute("templates", **job_kwargs)
sorting_analyzer.compute("noise_levels")
sorting_analyzer.compute("unit_locations", method="monopolar_triangulation")
sorting_analyzer.compute("isi_histograms")
sorting_analyzer.compute("correlograms", window_ms=100, bin_ms=5.)
sorting_analyzer.compute("principal_components", n_components=3, mode='by_channel_global', whiten=True, **job_kwargs)
sorting_analyzer.compute("quality_metrics", metric_names=["snr", "firing_rate"])
sorting_analyzer.compute("template_similarity")
sorting_analyzer.compute("spike_amplitudes", **job_kwargs)

In [ ]:
from spikeinterface.sorters import installed_sorters, run_sorter

installed_sorters() # ['kilosort4', 'lupin', 'simple', 'spykingcircus2', 'tridesclous2']

sorting_outputs_folder: Path = windows_to_wsl_path_if_needed(basedir.joinpath("SORTING").resolve()) # , f"{basename}-sorting_analyzer"
sorting_outputs_folder.mkdir(exist_ok=True)
print(f'sorting_outputs_folder: {sorting_outputs_folder}')


In [ ]:

# run SpykingCircus2
sorting_SC2 = run_sorter(sorter_name="spykingcircus2", recording=recording, folder=sorting_outputs_folder.joinpath("folder_SC2"), remove_existing_folder=True, delete_output_folder=True)


In [ ]:
# run Tridesclous
TDC_Output_Path: Path = sorting_outputs_folder.joinpath("folder_TDC")
sorting_TDC = run_sorter(sorter_name="tridesclous", recording=recording_filtered, folder=TDC_Output_Path)
sorting_TDC

In [ ]:
TDC_Final_Sorting_Output_Path: Path = TDC_Output_Path.joinpath('tridescloud_sorting_output')
sorting_TDC.save(folder=TDC_Final_Sorting_Output_Path)


In [ ]:
sorting_TDC.sorting_info

In [ ]:
# run Kilosort4
sorting_KS2_5 = run_sorter(sorter_name="kilosort4", recording=recording, folder=sorting_outputs_folder.joinpath("folder_KS4"))

In [ ]:
sorting = run_sorter(sorter_name="spykingcircus2", recording=recording, folder="/folder_SC2tmp/folder")

In [ ]:
# run SpykingCircus2
sorting_SC2 = run_sorter(sorter_name="spykingcircus2", recording=recording, folder=sorting_outputs_folder.joinpath("folder_SC2"))
sorting_SC2

In [ ]:
sorting_SC2


# Quality Metrics Tutorial

After spike sorting, you might want to validate the 'goodness' of the sorted units. This can be done using the
:code:`qualitymetrics` submodule, which computes several quality metrics of the sorted units.


In [ ]:
import spikeinterface.core as si
from spikeinterface.metrics import (
    compute_snrs,
    compute_presence_ratios,
    compute_isi_violations,
)

First, let's generate a simulated recording and sorting



In [ ]:
# sorting = sorting_TDC
# sorting_analyzer = sorting_TDC.to_shared_memory_sorting()

sorting_analyzer = si.create_sorting_analyzer(sorting=sorting, recording=recording_filtered)
sorting_analyzer.compute(['noise_levels','random_spikes','waveforms','templates','spike_locations','spike_amplitudes','correlograms','principal_components','quality_metrics','template_metrics'])


In [ ]:
sorting_analyzer.compute('template_metrics', include_multi_channel_metrics=True) 

# recording, sorting = si.generate_ground_truth_recording()
# print(recording)
# print(sorting)

In [ ]:
all_metric_names = list(sorting_analyzer.get_extension('quality_metrics').get_data().keys()) + list(sorting_analyzer.get_extension('template_metrics').get_data().keys())
print(set(model.feature_names_in_).issubset(set(all_metric_names)))

## Create SortingAnalyzer

For quality metrics we need first to create a :code:`SortingAnalyzer`.



The :code:`spikeinterface.qualitymetrics` submodule has a set of functions that allow users to compute
metrics in a compact and easy way. To compute a single metric, one can simply run one of the
quality metric functions as shown below. Each function has a variety of adjustable parameters that can be tuned.



In [ ]:
## INPUTS: sorting_analyzer
presence_ratios = compute_presence_ratios(sorting_analyzer)
print(presence_ratios)
isi_violation_ratio, isi_violations_count = compute_isi_violations(sorting_analyzer)
print(isi_violation_ratio)
snrs = compute_snrs(sorting_analyzer)
print(snrs)

In [ ]:
labels = sc.model_based_label_units(
    sorting_analyzer = sorting_analyzer,
    repo_id = "SpikeInterface/toy_tetrode_model",
    trusted = ['numpy.dtype']
)

print(labels)

To compute more than one metric at once, we can use the :code:`SortingAnalyzer.compute("quality_metrics")`
function and indicate which metrics we want to compute. Then we can retrieve the results using the :code:`get_data()`
method as a ``pandas.DataFrame``.



In [ ]:
metrics_ext = sorting_analyzer.compute(
    "quality_metrics",
    metric_names=["presence_ratio", "snr", "amplitude_cutoff"],
    metric_params={
        "presence_ratio": {"bin_duration_s": 2.0},
    }
)
metrics = metrics_ext.get_data()
print(metrics)

Some metrics are based on the principal component scores, so the extension
must be computed before. For instance:



In [ ]:
sorting_analyzer.compute("principal_components", n_components=5, mode="by_channel_global", whiten=True)

metrics_ext = sorting_analyzer.compute(
    "quality_metrics",
    metric_names=[
        "mahalanobis",
        "d_prime",
    ],
)
metrics = metrics_ext.get_data()
print(metrics)

# Trained Models

In [ ]:
from spikeinterface.curation import model_based_label_units

labels_and_probabilities = model_based_label_units(
    sorting_analyzer = sorting_analyzer,
    repo_id = "SpikeInterface/toy_tetrode_model",
    trust_model = True
)

# Apply to Phy Data

In [ ]:
import spikeinterface.curation as sc

## INPUTS: sorting = se.read_phy(phy_gui_dir)

job_kwargs = dict(n_jobs=8, progress_bar=True, chunk_duration="4s")

sorting_analyzer_folder = windows_to_wsl_path_if_needed(basedir.joinpath("spyk-circ", f"{basename}-phy-sorting_analyzer").resolve())

sorting_analyzer = si.create_sorting_analyzer(
    sorting=sorting,
    recording=recording_filtered,
    format="binary_folder",
    folder=sorting_analyzer_folder.as_posix(),
    overwrite=True,
)

In [ ]:
sorting_analyzer.compute([
    "noise_levels", "random_spikes", "waveforms", "templates",
    "spike_locations", "spike_amplitudes", "correlograms",
    "principal_components", "quality_metrics", "template_metrics",
], **job_kwargs)


In [ ]:
sorting_analyzer.compute("template_metrics", include_multi_channel_metrics=True)

In [ ]:
sorting_analyzer.save()

In [ ]:
model, model_info = sc.load_model(
    repo_id="SpikeInterface/UnitRefine_noise_neural_classifier",
    trust_model=True,
)

all_metric_names = (
    list(sorting_analyzer.get_extension("quality_metrics").get_data().keys())
    + list(sorting_analyzer.get_extension("template_metrics").get_data().keys())
)
print("model features:", model.feature_names_in_)
print("all present?:", set(model.feature_names_in_).issubset(set(all_metric_names)))
print("missing:", set(model.feature_names_in_) - set(all_metric_names))

In [ ]:
import numpy as np
import spikeinterface.curation.train_manual_curation as _tmc
import spikeinterface.curation.model_based_curation as _mbc

def _format_metric_dataframe_pandas15(input_data):
    input_data = input_data.applymap(lambda x: np.nan if np.isinf(x) else x)
    return input_data.astype("float32")

_tmc._format_metric_dataframe = _format_metric_dataframe_pandas15
_mbc._format_metric_dataframe = _format_metric_dataframe_pandas15

noise_neural_labels = sc.model_based_label_units(
    sorting_analyzer=sorting_analyzer,
    repo_id="SpikeInterface/UnitRefine_noise_neural_classifier",
    trust_model=True,
)

In [ ]:
noise_units = noise_neural_labels[noise_neural_labels["prediction"] == "noise"]
analyzer_neural = sorting_analyzer.remove_units(noise_units.index)

In [ ]:
# Step B: SUA vs MUA on remaining units
sua_mua_labels = sc.model_based_label_units(
    sorting_analyzer=analyzer_neural,
    repo_id="SpikeInterface/UnitRefine_sua_mua_classifier",
    trust_model=True,
)

In [ ]:
import pandas as pd
all_labels = pd.concat([sua_mua_labels, noise_units]).sort_index()
print(all_labels["prediction"].value_counts())
print(all_labels.head())

In [ ]:
phy_labels = sorting.get_property("quality")
comparison = all_labels.copy()
comparison["phy_label"] = phy_labels
comparison["agreement"] = comparison["prediction"] == comparison["phy_label"]
print(comparison.groupby(["phy_label", "prediction"]).size())

In [ ]:
good_units = all_labels[
    (all_labels["prediction"] == "sua")  # or map sua -> good for your pipeline
    & (all_labels["probability"] > 0.8)  # tune threshold
].index

sorting_curated = sorting.remove_units(
    remove_unit_ids=[u for u in sorting.get_unit_ids() if u not in good_units]
)

In [ ]:
good_units

In [ ]:
# sua -> good, mua -> mua, noise -> noise
label_map = {"sua": "good", "mua": "mua", "neural": "unclassified", "noise": "noise"}
sorting.set_property("auto_quality", [label_map.get(p, "unclassified") for p in all_labels.loc[sorting.get_unit_ids(), "prediction"]])

In [ ]:
sorting

In [ ]:
metrics_df = sorting_analyzer.get_extension("quality_metrics").get_data()
template_df = sorting_analyzer.get_extension("template_metrics").get_data()

out_dir = sorting_analyzer_folder / "exports"
out_dir.mkdir(exist_ok=True)
metrics_df.to_csv(out_dir / "quality_metrics.csv")
template_df.to_csv(out_dir / "template_metrics.csv")

In [ ]:
metrics_df

In [ ]:
template_df

## Write out labels to Phy

In [38]:
import shutil
import pandas as pd

phy_gui_dir = windows_to_wsl_path_if_needed(
    basedir.joinpath("spyk-circ", basename, f"{basename}-merged.GUI").resolve()
)
cluster_info_path = phy_gui_dir / "cluster_info.tsv"

if not (cluster_info_path.exists() and cluster_info_path.is_file()):
    print(f'WARNING: phy has not been launched at least once so cluster_info_path: {cluster_info_path} does not yet exist. Launch phy to create this file.')
    
else:
    # Backup first
    shutil.copy(cluster_info_path, cluster_info_path.with_suffix(".tsv.bak"))

    cluster_info = pd.read_csv(cluster_info_path, sep="\t")

    label_map = {"sua": "good", "mua": "mua", "noise": "noise"}
    phy_groups = all_labels["prediction"].map(label_map).fillna("")

    # unit_ids from read_phy match cluster_id in cluster_info.tsv
    cluster_info["group"] = cluster_info["cluster_id"].map(phy_groups).fillna(cluster_info["group"])

    # Optional: keep model outputs as extra columns (visible in Phy Cluster View)
    cluster_info["model_prediction"] = cluster_info["cluster_id"].map(all_labels["prediction"])
    cluster_info["model_probability"] = cluster_info["cluster_id"].map(all_labels["probability"])

    cluster_info.to_csv(cluster_info_path, sep="\t", index=False)